# Error analysis & ablative analysis

_Machine Learning — more_

**Find which part of your system to fix — and which part actually earns its keep.**

Real systems are pipelines: several components in a row (preprocess → detect → classify …).

     When the whole thing underperforms, where do you spend your effort? Guessing wastes weeks.

---

This notebook is a practice scaffold for the **Error analysis & ablative analysis** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

**Ablative analysis** asks the opposite of feature selection: start from the full system, then *remove* one block of features at a time and see how much accuracy you lose. The block whose removal hurts the most is the one earning its keep. We build this in three steps: (1) a synthetic dataset and a scoring helper, (2) the full-system baseline, (3) the ablation loop.

### Step 1 — Make the data and a scoring helper

We generate a synthetic 8-feature classification problem so we know the ground truth. The `acc` helper trains a logistic-regression model on whatever subset of columns we hand it and returns its 5-fold cross-validated accuracy — a single, comparable number for any feature subset.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Synthetic data: 400 examples, 8 features, 4 of them genuinely informative.
X, y = make_classification(n_samples=400, n_features=8, n_informative=4,
                           n_redundant=0, random_state=0)

# Helper: 5-fold cross-validated accuracy using only the given columns.
def acc(cols):
    model = LogisticRegression(max_iter=1000)
    scores = cross_val_score(model, X[:, cols], y, cv=5)
    return scores.mean()

### Step 2 — Measure the full-system baseline

First we score the system with **all 8 features** present. Every ablation below is measured as a *drop* relative to this baseline, so we need it before removing anything.

In [ ]:
all_columns = list(range(8))
full = acc(all_columns)                 # baseline: every feature present
print("full system accuracy = %.4f" % full)

### Step 3 — Ablate each block and find the most important

We group the features into four blocks. For each block we rebuild the system *without* it (keeping every other column), re-score, and record how much accuracy was lost. The block with the largest drop is the one the system most depends on.

In [ ]:
# Group the 8 features into four blocks of two.
blocks = {"feat0-1": [0, 1], "feat2-3": [2, 3],
          "feat4-5": [4, 5], "feat6-7": [6, 7]}

drops = {}
for name, cols in blocks.items():
    keep = [c for c in range(8) if c not in cols]   # all columns except this block
    drop = full - acc(keep)                          # accuracy lost without the block
    drops[name] = drop
    print("remove %-8s -> drop = %+.4f" % (name, drop))

top = max(drops, key=drops.get)
print("most important block = %s (largest drop)" % top)

## Visualize the data & results

_Which block of chemical features actually earns its keep?_

Now we run the same ablation idea on **real data** — the Wine dataset — and draw the result as a bar chart. We build it in three steps: (1) load and standardise the real data, (2) ablate each chemical-feature block, (3) plot the accuracy lost per block.

### Step 1 — Load and standardise the real Wine data

The Wine dataset has 178 wines, 13 chemical measurements, and 3 cultivars. We standardise the features (zero mean, unit variance) so logistic regression treats every measurement on a comparable scale, then reuse the same cross-validated `acc` helper pattern.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# REAL data: 178 wines, 13 chemical features, 3 cultivars.
wn = load_wine()
X = StandardScaler().fit_transform(wn.data)
y = wn.target

def acc(cols):
    model = LogisticRegression(max_iter=5000)
    scores = cross_val_score(model, X[:, cols], y, cv=5)
    return scores.mean()

### Step 2 — Ablate each chemical-feature block

We group the 13 measurements into five chemically meaningful blocks and, exactly as before, remove each block in turn and record the accuracy lost. We collect the block names and their drops into parallel lists ready for plotting.

In [ ]:
full = acc(list(range(13)))              # baseline: all 13 features

blocks = {'alcohol/acid': [0, 1, 2], 'ash/Mg': [3, 4, 5],
          'phenols': [6, 7, 8], 'color/hue': [9, 10, 11], 'proline': [12]}

names = []
drops = []
for name, cols in blocks.items():
    keep = [c for c in range(13) if c not in cols]
    names.append(name)
    drops.append(full - acc(keep))       # accuracy lost without that block

### Step 3 — Plot the accuracy lost per block

We draw one bar per block, its height the accuracy lost (in percent) when that block is removed. The single most important block — the largest drop — is highlighted in orange so it stands out at a glance.

In [ ]:
# Highlight the most important block (the largest drop) in orange.
max_drop = max(drops)
colors = ['#ffb454' if d == max_drop else '#4ea1ff' for d in drops]

drops_percent = [d * 100 for d in drops]
plt.bar(names, drops_percent, color=colors)
plt.ylabel('accuracy lost (%)')
plt.title('Wine: accuracy lost when each chemical feature block is removed')
plt.show()